# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-4o-mini'
openai = OpenAI()

API key looks good so far


In [4]:
# A class to represent a Webpage

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [9]:
ed = Website("https://edwarddonner.com")
ed.links

['https://edwarddonner.com/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://patents.google.com/patent/US20210049536A1/',
 'https://www.linkedin.com/in/eddonner/',
 'https://edwarddonner.com/2025/01/23/llm-workshop-hands-on-with-agents-resources/',
 'https://edwarddonner.com/2025/01/23/llm-workshop-hands-on-with-agents-resources/',
 'https://edwarddonner.com/2024/12/21/llm-resources-superdatascience/',
 'https://edwarddonner.com/2024/12/21/llm-resources-superdatascience/',
 'https://edwarddonner.com/2024/11/13/llm-engineering-resources/',
 'https://edwarddonner.com/2024/11/13/llm-engineering-resources/',
 'ht

## First step: Have GPT-4o-mini figure out which links are relevant

### Use a call to gpt-4o-mini to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [10]:
link_system_prompt = "You are provided with a list of links found on a webpage. \
You are able to decide which of the links would be most relevant to include in a brochure about the company, \
such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
}
""" #one-shot content

In [11]:
print(link_system_prompt)

You are provided with a list of links found on a webpage. You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
}



In [12]:
def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [13]:
print(get_links_user_prompt(ed))

Here is the list of links on the website of https://edwarddonner.com - please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. Do not include Terms of Service, Privacy, email links.
Links (some might be relative links):
https://edwarddonner.com/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://patents.google.com/patent/US20210049536A1/
https://www.linkedin.com/in/eddonner/
https://edwarddonner.com/2025/01/23/llm-workshop-hands-on-with-agents-resources/
https://edwarddonner.com/2025/01/23/llm-workshop-hands-on-with-agents-resources/
https://edwarddonner.com/2024/12/21/

In [27]:
def get_links(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [23]:
# Anthropic has made their site harder to scrape, so I'm using HuggingFace..

huggingface = Website("https://huggingface.co")
huggingface.links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/posts',
 '/docs',
 '/enterprise',
 '/pricing',
 '/login',
 '/join',
 '/spaces',
 '/models',
 '/meta-llama/Llama-4-Scout-17B-16E-Instruct',
 '/agentica-org/DeepCoder-14B-Preview',
 '/reducto/RolmOCR',
 '/meta-llama/Llama-4-Maverick-17B-128E-Instruct',
 '/deepseek-ai/DeepSeek-V3-0324',
 '/models',
 '/spaces/enzostvs/deepsite',
 '/spaces/jamesliu1217/EasyControl_Ghibli',
 '/spaces/VAST-AI/TripoSG',
 '/spaces/Stable-X/Hi3DGen',
 '/spaces/victor/deepsite-gallery',
 '/spaces',
 '/datasets/nvidia/OpenCodeReasoning',
 '/datasets/open-thoughts/OpenThoughts2-1M',
 '/datasets/nvidia/Llama-Nemotron-Post-Training-Dataset',
 '/datasets/agentica-org/DeepCoder-Preview-Dataset',
 '/datasets/OmniSVG/MMSVG-Illustration',
 '/datasets',
 '/join',
 '/pricing#endpoints',
 '/pricing#spaces',
 '/pricing',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/allenai',
 '/facebook',
 '/amazon',
 '/google'

In [24]:
get_links("https://huggingface.co")

{'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'developers page', 'url': 'https://huggingface.co/docs'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT4-o

In [28]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Found links:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [32]:
#print(get_all_details("https://huggingface.co"))

In [31]:
system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
Include details of company culture, customers and careers/jobs if you have the information."

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
# and creates a short humorous, entertaining, jokey brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
# Include details of company culture, customers and careers/jobs if you have the information."


In [33]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [34]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'company page', 'url': 'https://www.linkedin.com/company/huggingface/'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'documentation page', 'url': 'https://huggingface.co/docs'}]}


'You are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\nLanding page:\nWebpage Title:\nHugging Face – The AI community building the future.\nWebpage Contents:\nHugging Face\nModels\nDatasets\nSpaces\nPosts\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 1M+ models\nTrending on\nthis week\nModels\nmeta-llama/Llama-4-Scout-17B-16E-Instruct\nUpdated\n2 days ago\n•\n338k\n•\n719\nagentica-org/DeepCoder-14B-Preview\nUpdated\n1 day ago\n•\n4.98k\n•\n339\nreducto/RolmOCR\nUpdated\n9 days ago\n•\n4.39k\n•\n303\nmeta-llama/Llama-4-Maverick-17B-128E-Instruct\nUpdated\n2 days ago\n•\n24.2k\n•\n276\ndeepseek-ai/DeepSeek-V3-0324\nUpdated\n15 days ago\n•\n189k\n•\n2.52k\nBrowse 1M+ models\nSpaces\nR

In [35]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [36]:
create_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/about'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'company page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'linkedin page', 'url': 'https://www.linkedin.com/company/huggingface/'}, {'type': 'twitter page', 'url': 'https://twitter.com/huggingface'}, {'type': 'discussion page', 'url': 'https://discuss.huggingface.co'}]}


# Hugging Face Brochure

## Welcome to Hugging Face
**The AI community building the future.**  
At Hugging Face, we provide a collaborative platform where the machine learning community can unite, innovate, and transform ideas into reality. Whether you’re a developer, researcher, or industry leader, you’ll find a vibrant ecosystem encompassing state-of-the-art models, expansive datasets, and versatile applications.

---

## Our Offerings

### **Models & Datasets**
We host and maintain over **1 million models** and **250k datasets**, providing the tools to create, discover, and collaborate on cutting-edge machine learning projects. Explore trending models such as **Llama-4-Scout** and **DeepCoder-Preview**, or dive into our rich library of datasets tailored for various ML tasks.

### **Spaces**
Hugging Face offers the ability to deploy applications seamlessly. Our **Spaces** feature allows users to run applications like **DeepSite** for application generation, and Hi3DGen for high-fidelity 3D geometry generation from images.

### **Enterprise Solutions**
For teams seeking advanced capabilities, our **Enterprise program** offers optimized performance with features such as dedicated support, single sign-on, and enterprise-grade security. Join over **50,000 organizations** that trust Hugging Face to power their AI initiatives, including major players like Google, Amazon, and Microsoft.

---

## Company Culture

At Hugging Face, we foster a culture of **openness and collaboration**. Our community-driven approach encourages contributions from developers and researchers across the globe. 

### **Our Values**
- **Open Source**: We believe in building the foundation of machine learning together with the community. Explore our contributions through tools like Transformers, Diffusers, and Tokenizers.
- **Innovation**: With a focus on advancing technology, we continually push the boundaries of what AI can achieve.
- **Inclusivity**: We celebrate diverse backgrounds and perspectives, ensuring a welcoming environment for all members of our community.

---

## Careers at Hugging Face

Join a team that is reshaping the future of AI! We offer exciting career opportunities across various fields, from machine learning research to software engineering. Our commitment to employee development includes competitive salaries, flexible working options, and a supportive, growth-oriented atmosphere.

Check out our **[Current Openings](#)** to see how you can contribute to making AI accessible and effective for everyone.

---

## Get Involved

Whether you are a prospective customer, an investor, or a potential recruit, Hugging Face has something for you.

### **Contact Us**
- **Website**: [Hugging Face](https://huggingface.co)
- **Follow Us**: [GitHub](https://github.com/huggingface) | [Twitter](https://twitter.com/huggingface) | [LinkedIn](https://linkedin.com/company/huggingface)

Together, let’s build the future of AI!

---  
*This brochure aims to provide an overview of Hugging Face for prospective customers, investors, and recruits. For more detailed inquiries or specific offers, please explore our website.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [37]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

    #for chunk in stream:
    #    print(chunk.choices[0].delta.content or '', end='')

In [38]:
stream_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'community discussion page', 'url': 'https://discuss.huggingface.co'}, {'type': 'GitHub page', 'url': 'https://github.com/huggingface'}, {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/company/huggingface'}, {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}]}


# Hugging Face Brochure

**Company Overview**
Hugging Face is a pioneering leader in the artificial intelligence community, dedicated to shaping the future of machine learning. As a collaborative platform, Hugging Face empowers users to explore, create, share, and collaborate on models, datasets, and applications, fostering a rich ecosystem for AI enthusiasts and professionals alike.

---

## Our Mission

At Hugging Face, we believe in community-driven development and open-source innovation. Our platform serves as a foundation for building the tools necessary to accelerate machine learning research and applications, ultimately contributing to the advancement of AI technology for all.

## What We Offer

- **Models**: Access a vast library with over 1 million models, including state-of-the-art solutions for various tasks like text, image, video, and audio processing.
- **Datasets**: A collection of over 250,000 datasets tailored for machine learning tasks, facilitating easy access and sharing of data for research and development.
- **Spaces**: Discover and create applications through our collaborative Spaces environment, hosting over 400,000 applications designed by the community.
- **Enterprise Solutions**: Offering tailored solutions for organizations to enhance their AI capabilities, featuring enterprise-grade security and dedicated support.

---

## Community & Culture

Hugging Face thrives on a vibrant company culture that promotes inclusivity, innovation, and teamwork. We currently host more than 50,000 organizations, including industry leaders such as Google, Amazon, and Microsoft. Our community is built on collaboration, where both novice and experienced users can contribute to ongoing projects and share their achievements. We value creativity, encourage the sharing of ideas, and support continuous learning, making it an ideal environment for AI and machine learning professionals.

## Career Opportunities

Join our dynamic team and become part of a revolutionary movement in the AI landscape. Hugging Face offers rewarding career paths across various domains including:

- Machine Learning Engineering
- Data Science
- Product Management
- Software Development
- Community Engagement

We embrace diversity and are on the lookout for passionate individuals who can add value to our mission, continue to drive technological advancements, and enhance the AI community.

### Current Openings

- Software Engineer
- Data Scientist
- Community Manager
- Product Marketing Manager

Visit our [Careers Page](https://huggingface.co/jobs) to explore available opportunities and how you can contribute to building the future of AI.

---

## Connect with Us

Stay updated with the latest trends, models, and community contributions by following us on our social channels:

- [GitHub](https://github.com/huggingface)
- [Twitter](https://twitter.com/huggingface)
- [LinkedIn](https://linkedin.com/company/huggingface)
- [Discord](https://discord.gg/huggingface)

Join us in our mission to democratize AI and unlock its potential for everyone. Together, let's build the future!

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>

In [46]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

## How to create and OpenAI Key
1. Create an OpenAI Account: https://platform.openai.com/signup
2. Navigate to https://platform.openai.com/account/api-keys
3. Click on "Create new secrete key"
4. Save the key in an empty .env file in the root directory of the project and add: OPENAI_API_KEY=your key

Note: you may have to pay a minimum credit (5$) to be able to use openai models.


In [47]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-4o-mini'
openai = OpenAI()

API key looks good so far


In [48]:
# A class to represent a Webpage

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [54]:
class CreateBrochure:
    def __init__(
        self,
        model,
        company_name,
        url,
        language
    ):

        #super().__init__(self, url)

        self.model = model
        self.company_name = company_name
        self.website = Website(url)
        self.language = language

        self.link_system_prompt = "You are provided with a list of links found on a webpage. \
        You are able to decide which of the links would be most relevant to include in a brochure about the company, \
        such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
        self.link_system_prompt += "You should respond in JSON as in this example:"
        self.link_system_prompt += """
        {
            "links": [
                {"type": "about page", "url": "https://full.url/goes/here/about"},
                {"type": "careers page": "url": "https://another.full.url/careers"}
            ]
        }
        """
        
        self.link_user_prompt = f"Here is the list of links on the website of {self.website.url} - "
        self.link_user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
        Do not include Terms of Service, Privacy, email links.\n"
        self.link_user_prompt += "Links (some might be relative links):\n"
        self.link_user_prompt += "\n".join(self.website.links)

        self.brochure_system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
        and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
        Include details of company culture, customers and careers/jobs if you have the information."
        self.brochure_system_prompt += f"If the brochure is not in {self.language}, translate it into {self.language} as the final step. "
        self.brochure_system_prompt += "Do not translate the company name."
        self.brochure_system_prompt += "Only output the translated version."

        self.brochure_user_prompt = f"You are looking at a company called: {self.company_name}\n"
        self.brochure_user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown and in {self.language}.\n"
        self.brochure_user_prompt += self.get_all_details()
        self.brochure_user_prompt = self.brochure_user_prompt[:5_000] # Truncate if more than 5,000 characters
        

    def get_links(self):        
        response = openai.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": self.link_system_prompt},
                {"role": "user", "content": self.link_user_prompt}
          ],
            response_format={"type": "json_object"}
        )
        result = response.choices[0].message.content
        return json.loads(result)

    def get_all_details(self):
        result = "Landing page:\n"
        result += self.website.get_contents()
        links = self.get_links()
        #print("Found links:", links)
        for link in links["links"]:
            result += f"\n\n{link['type']}\n"
            result += Website(link["url"]).get_contents()
        return result

    def display(self):
        stream = openai.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": self.brochure_system_prompt},
                {"role": "user", "content": self.brochure_user_prompt}
              ],
            stream=True
        )

        response = ""
        display_handle = display(Markdown(""), display_id=True)
        for chunk in stream:
            response += chunk.choices[0].delta.content or ''
            response = response.replace("```","").replace("markdown", "")
            update_display(Markdown(response), display_id=display_handle.display_id)

In [55]:
brochure = CreateBrochure(
    model=MODEL,
    company_name="HuggingFace",
    url="https://huggingface.co",
    language="Spanish")
brochure.display()
    


# Brochure de Hugging Face

## ¡La comunidad de IA construyendo el futuro!

### ¿Quiénes somos?
Hugging Face es la plataforma colaborativa donde la comunidad de aprendizaje automático trabaja en modelos, conjuntos de datos y aplicaciones. Somos pioneros en el desarrollo de herramientas de inteligencia artificial y diseño colaborativo.

### ¿Qué ofrecemos?
- **Modelos:** Acceso a más de 1 millón de modelos de IA.
- **Conjuntos de Datos:** Más de 250,000 datasets disponibles para facilitar el aprendizaje automático.
- **Espacios:** Crea y ejecuta aplicaciones de IA en nuestra plataforma.
- **Open Source:** Contribuye y aprovecha nuestras herramientas como Transformers, Diffusers, y más.

### Nuestros clientes
Más de 50,000 organizaciones ya utilizan Hugging Face, incluyendo gigantes como:
- **Amazon**
- **Google**
- **Microsoft**
- **Meta**

### Cultura de la empresa
En Hugging Face, fomentamos un ambiente colaborativo y abierto. Valorizamos la diversidad, la creatividad y el trabajo en equipo, buscando siempre la innovación y la mejora continua en nuestras herramientas y servicios.

### Carreras en Hugging Face
Estamos buscando talento apasionado por la inteligencia artificial y el aprendizaje automático. Únete a nuestro equipo y ayuda a construir un futuro mejor a través de la IA.
- **Posiciones abiertas:** Explora oportunidades en ingeniería, gestión de productos, marketing y más en nuestra sección de **Jobs**.

### Conéctate con nosotros
Mantente al día con nuestras novedades y únete a nuestra comunidad:
- **GitHub**
- **Twitter**
- **LinkedIn**
- **Discord**

Visita [huggingface.co](https://huggingface.co) para más información y comienza tu viaje con el futuro de la inteligencia artificial hoy.

